In [ ]:
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

# Clean up old test directories for a fresh run
for path in [
    "/tmp/stream_input",
    "/tmp/spark_checkpoints/transactions_stream",
]:
    shutil.rmtree(path, ignore_errors=True)
os.makedirs("/tmp/stream_input", exist_ok=True)

# 1. Create sample JSON input files
with open("/tmp/stream_input/batch1.json", "w") as f:
    f.write(
        '{"transaction_id": "t1", "user_id": "u1", "amount": 50.0, "event_time": "2026-11-16 10:00:00"}\n'
    )
    f.write(
        '{"transaction_id": "t2", "user_id": "u1", "amount": 25.0, "event_time": "2026-11-16 10:02:00"}\n'
    )
    f.write(
        '{"transaction_id": "t3", "user_id": "u2", "amount": 80.0, "event_time": "2026-11-16 10:15:00"}\n'
    )

# 2. Initialize SparkSession
spark = (
    SparkSession.builder.appName("FaultTolerantFileStream")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# 3. Define Schema
event_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time", StringType(), True),
])

# 4. Read Stream from Directory
raw_stream_df = (
    spark.readStream.format("json")
    .schema(event_schema)
    .load("/tmp/stream_input")
)

parsed_df = raw_stream_df.withColumn(
    "event_timestamp", col("event_time").cast("timestamp")
)

# 5. Watermarking & Windowed Aggregation
aggregated_df = (
    parsed_df.withWatermark("event_timestamp", "10 minutes")
    .groupBy(window(col("event_timestamp"), "5 minutes"), col("user_id"))
    .sum("amount")
    .withColumnRenamed("sum(amount)", "total_spent")
)

# 6. Write to 'memory' sink so Jupyter can display the table directly
query = (
    aggregated_df.writeStream.outputMode("complete")
    .format("memory")
    .queryName("transactions_table")
    .option("checkpointLocation", "/tmp/spark_checkpoints/transactions_stream")
    .trigger(
        availableNow=True
    )  # Processes all available files cleanly without arbitrary timeouts
    .start()
)

query.awaitTermination()

# 7. Display the output in the Jupyter cell
spark.sql("SELECT * FROM transactions_table ORDER BY user_id").show(
    truncate=False
)

+------------------------------------------+-------+-----------+
|window                                    |user_id|total_spent|
+------------------------------------------+-------+-----------+
|{2026-11-16 10:00:00, 2026-11-16 10:05:00}|u1     |75.0       |
|{2026-11-16 10:15:00, 2026-11-16 10:20:00}|u2     |80.0       |
+------------------------------------------+-------+-----------+



In [ ]:
from collections import defaultdict
import hashlib
import os
import shutil

# 1. Setup a test directory with sample files (including duplicates)
test_dir = "/tmp/duplicate_check_demo"
shutil.rmtree(test_dir, ignore_errors=True)
os.makedirs(f"{test_dir}/subfolder", exist_ok=True)

# Create two identical files with different names
with open(f"{test_dir}/report_v1.txt", "w") as f:
    f.write("Quarterly revenue is $1.2M.\nAll targets met.")

with open(f"{test_dir}/subfolder/report_copy.txt", "w") as f:
    f.write("Quarterly revenue is $1.2M.\nAll targets met.")

# Create a third file with the exact same byte size, but different content
with open(f"{test_dir}/report_v2.txt", "w") as f:
    f.write("Quarterly revenue is $1.5M.\nAll targets met.")

# Create a completely unique file
with open(f"{test_dir}/notes.txt", "w") as f:
    f.write("Meeting notes from Monday.")


# 2. Hashing and Duplicate Detection Functions
def get_file_hash(filepath, chunk_size=65536):
    """Calculates SHA-256 hash of a file in memory-safe chunks."""
    sha256 = hashlib.sha256()
    try:
        with open(filepath, "rb") as f:
            while chunk := f.read(chunk_size):
                sha256.update(chunk)
        return sha256.hexdigest()
    except (OSError, PermissionError) as e:
        print(f"Skipping {filepath}: {e}")
        return None


def find_duplicate_files(directory_path):
    """Finds duplicate files using file-size grouping + SHA-256 verification."""
    files_by_size = defaultdict(list)

    # Pass 1: Group files by exact byte size
    for root, _, files in os.walk(directory_path):
        for filename in files:
            filepath = os.path.join(root, filename)
            if not os.path.islink(filepath):
                try:
                    files_by_size[os.path.getsize(filepath)].append(filepath)
                except OSError:
                    continue

    # Pass 2: Compute SHA-256 only for files that share a file size
    duplicates = defaultdict(list)
    for file_size, paths in files_by_size.items():
        if len(paths) < 2:
            continue
        for path in paths:
            file_hash = get_file_hash(path)
            if file_hash:
                duplicates[file_hash].append(path)

    return {h: paths for h, paths in duplicates.items() if len(paths) > 1}


# 3. Run and display results
duplicate_groups = find_duplicate_files(test_dir)

if not duplicate_groups:
    print("No duplicate files found.")
else:
    print(f"Found {len(duplicate_groups)} set(s) of duplicate files:\n")
    for file_hash, paths in duplicate_groups.items():
        print(f"SHA-256: {file_hash[:16]}...")
        for p in paths:
            print(f"  - {p}")

Found 1 set(s) of duplicate files:

SHA-256: f489e1e2d30f594a...
  - /tmp/duplicate_check_demo/report_v1.txt
  - /tmp/duplicate_check_demo/subfolder/report_copy.txt
